# GeoLife 1.3 (Beijing) → TTE format

**Source:** Microsoft Research, `microsoft.com/en-us/download/details.aspx?id=52367`
(direct download, no login; a Kaggle mirror exists). **298.7 MB, 17 621
trajectories, 182 users, Apr 2007 – Aug 2012**, 91 % of it logged every 1–5 s or
5–10 m. **No licence is stated** — cite Zheng et al. and ship the loader.

The unique property: **69 of the users labelled their trajectories with the mode
of transport** (`walk`, `bike`, `bus`, `car`, `taxi`, `subway`, `train`, …).
That makes GeoLife the only open source on which "does a TTE model trained on
cars transfer to buses?" can be asked at all.

**Layout:**

```
Geolife Trajectories 1.3/Data/<user>/Trajectory/<yyyyMMddHHmmss>.plt
Geolife Trajectories 1.3/Data/<user>/labels.txt          # 69 users only
```

`.plt` — 6 header lines, then `lat,lon,0,altitude,days_since_1899-12-30,date,time`.
`labels.txt` — tab separated, `Start Time	End Time	Transportation Mode`.

Both clocks are the same clock, so the label join is exact; what that clock *is*
(GMT vs Beijing local) is stated as GMT in the user guide — set `SOURCE_TZ` if
your copy disagrees, it only shifts the absolute epochs, never the durations.

## Target format (the "gold" contract)

Taken from the Harbin files in `datasets.zip`, with the Omsk file naming:

| file | contents |
|---|---|
| `matched_trips_<city>.csv` | unnamed index, `Id`, `Coordinates`, `OSMids`, `Timestamps`, `Total_time` |
| `edge_list_directed_<city>.csv` | `osmid_u`, `osmid_v` — directed transitions between road segments |
| `road_network_unique_osmids_<city>.geojson` | one `LineString` per segment, `osmid` / `original_osmid` / `is_duplicate` / `duplicate_index` / `length` / `highway` / ... |

`Coordinates`, `OSMids` and `Timestamps` are Python-literal lists of **equal
length — one entry per GPS fix**: `(lon, lat)` floats, the segment id the fix was
matched to (a string), and the unix timestamp in seconds.
`Total_time = Timestamps[-1] - Timestamps[0]`, in seconds.

In [ ]:
CITY    = "beijing_geolife"
RAW_DIR = "Geolife Trajectories 1.3/Data"
OUT     = "."
GRAPHML = "beijing_drive.graphml"

BBOX = (116.20, 39.75, 116.55, 40.05)     # Beijing inside the 5th ring
SOURCE_TZ = "UTC"                          # per the user guide the .plt clock is GMT

# Which labelled modes to keep.  Road-vehicle modes only by default; add
# "bike"/"walk" if the cross-mode transfer experiment is the point.
MODES = ["car", "taxi", "bus"]
USE_UNLABELLED = False    # True also segments the 113 unlabelled users by gap/standstill

MAX_TRIPS   = 20000
RANDOM_SEED = 0

MIN_POINTS       = 8
MIN_SECONDS      = 120
MAX_SECONDS      = 5400
MIN_METERS       = 800
MAX_SPEED_MS     = 45.0
MEAN_SPEED_RANGE = (1.0, 25.0)

MAX_GAP_S        = 300
STILL_M          = 25
STOP_S           = 180

SNAP_RADIUS_M    = 60
SNAP_SIGMA_M     = 20
MAX_SNAP_M       = 50
MIN_MATCHED_FRAC = 0.8
MIN_CONNECTIVITY = 0.99       # after route filling this should be exactly 1.0
BETA_M           = 10.0       # transition scale: |network dist - straight dist| / beta
MAX_ROUTE_FACTOR = 4.0        # reject a detour longer than this x the straight gap
MAX_MEDIAN_GAP_S = 20    # drop trips whose fixes are further apart in time

In [ ]:
# pip install pandas numpy osmnx geopandas shapely networkx tqdm
import ast, json, math, os, glob
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import Point, mapping
from pyproj import Transformer
from tqdm.auto import tqdm

## Helpers

In [ ]:
# --- unique OSM ids -----------------------------------------------------------
def assign_unique_osmids(G):
    """(u, v, key) -> {unique_osmid, original_osmid, is_duplicate, duplicate_index}.

    One OSM way is split into many graph edges, so `osmid` is not unique.  The
    gold files use `<way_id>` when a way appears once and `<way_id>_<i>`
    (i = 1, 2, ...) for every edge of a way that appears several times -- see
    `Harbin_edge_list.csv` ('858780553' next to '705148575_1').
    """
    base = {}
    for u, v, k, d in G.edges(keys=True, data=True):
        o = d.get("osmid")
        if isinstance(o, (list, tuple, set)):
            o = sorted(o)[0]
        base[(u, v, k)] = str(o)

    counts = pd.Series(list(base.values())).value_counts().to_dict()
    seen, out = {}, {}
    for e, b in base.items():
        if counts[b] == 1:
            out[e] = dict(unique_osmid=b, original_osmid=b,
                          is_duplicate=False, duplicate_index=0)
        else:
            i = seen.get(b, 0) + 1
            seen[b] = i
            out[e] = dict(unique_osmid=f"{b}_{i}", original_osmid=b,
                          is_duplicate=True, duplicate_index=i)
    return out


# --- edge_list_directed_<city>.csv --------------------------------------------
def build_edge_list_directed(G, uid):
    """Directed transitions between segments that share a node (Harbin style)."""
    inc, out = {}, {}
    for u, v, k in G.edges(keys=True):
        out.setdefault(u, []).append((u, v, k))
        inc.setdefault(v, []).append((u, v, k))
    rows = set()
    for node in set(inc) & set(out):
        for e1 in inc[node]:
            for e2 in out[node]:
                if e1 == e2:
                    continue
                a, b = uid[e1]["unique_osmid"], uid[e2]["unique_osmid"]
                if a != b:
                    rows.add((a, b))
    return pd.DataFrame(sorted(rows), columns=["osmid_u", "osmid_v"])


# --- road_network_unique_osmids_<city>.geojson --------------------------------
_KEEP = ["bridge", "highway", "lanes", "name", "oneway", "reversed",
         "junction", "ref", "tunnel", "maxspeed", "access", "width"]


def _clean(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None
    if isinstance(val, (list, tuple)):
        val = [str(x) for x in val]
        return val or None
    return val if isinstance(val, bool) else str(val)


def build_road_network_geojson(G, uid, path):
    """One LineString per graph edge with the Harbin property set."""
    Gu = G if str(G.graph.get("crs", "")).lower() in ("epsg:4326", "wgs84") \
        else ox.projection.project_graph(G, to_latlong=True)
    edges = ox.convert.graph_to_gdfs(Gu, nodes=False, edges=True, fill_edge_geometry=True)
    feats = []
    for (u, v, k), row in edges.iterrows():
        info = uid[(u, v, k)]
        props = {"u": int(u), "v": int(v), "key": int(k),
                 "osmid": info["original_osmid"],
                 "unique_osmid": info["unique_osmid"]}
        for c in _KEEP:
            props[c] = _clean(row[c]) if c in edges.columns else None
        props["length"] = float(row["length"])
        props["original_osmid"] = info["original_osmid"]
        props["is_duplicate"] = info["is_duplicate"]
        props["duplicate_index"] = info["duplicate_index"]
        props = {a: b for a, b in props.items() if b is not None}
        feats.append({"type": "Feature", "properties": props,
                      "geometry": mapping(row["geometry"])})
    fc = {"type": "FeatureCollection", "name": "road_network_unique_osmids",
          "crs": {"type": "name",
                  "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
          "features": feats}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False)
    return len(feats)


# --- map matching -------------------------------------------------------------
class Matcher:
    """HMM map matcher that returns a *connected route*, not one edge per fix.

    emission   = snap distance, Gaussian with `sigma_m`
    transition = |network distance - straight-line distance| / `beta_m`
                 (Newson & Krumm)

    The old hop-count transition (same / 1-hop / 2-hop) silently assumed the
    vehicle never travelled more than two edges between two fixes.  That holds
    at 1-7 s sampling and breaks completely at 30-60 s, where a car crosses
    half a dozen blocks between fixes.  A distance-based transition has no such
    assumption.

    `match_route` also inserts the edges the vehicle must have crossed between
    two fixes, so consecutive segments in the output are genuinely adjacent and
    `connectivity()` measures something real instead of the sampling rate.
    """

    def __init__(self, G, radius_m=60.0, sigma_m=20.0, k_candidates=8,
                 beta_m=10.0, max_route_factor=4.0, min_route_slack_m=300.0):
        projected = str(G.graph.get("crs", "")).lower() not in ("epsg:4326", "wgs84", "")
        self.Gp = G if projected else ox.projection.project_graph(G)
        self.crs = self.Gp.graph["crs"]
        self.edges = ox.convert.graph_to_gdfs(self.Gp, nodes=False, edges=True,
                                              fill_edge_geometry=True)
        self.index = list(self.edges.index)
        self.sindex = self.edges.sindex
        self.geom = dict(zip(self.index, self.edges.geometry))
        self.length = {e: float(l) for e, l in zip(self.index, self.edges["length"])}
        self.radius, self.sigma, self.k = radius_m, sigma_m, k_candidates
        self.beta = beta_m
        self.max_factor, self.slack = max_route_factor, min_route_slack_m
        self.succ = {}
        for u, v, k in self.Gp.edges(keys=True):
            self.succ.setdefault(u, []).append((u, v, k))
        self._to_lonlat = Transformer.from_crs(self.crs, "EPSG:4326",
                                               always_xy=True).transform
        self._dij = {}

    # ---------------------------------------------------------------- helpers
    def _dijkstra(self, source, cutoff):
        """Cached single-source distances; recomputed only if the cutoff grows."""
        hit = self._dij.get(source)
        if hit is None or hit[0] < cutoff:
            dist = nx.single_source_dijkstra_path_length(
                self.Gp, source, cutoff=cutoff, weight="length")
            self._dij[source] = (cutoff, dist)
            return dist
        return hit[1]

    def _net_dist(self, e1, e2, cutoff):
        if e1 == e2 or e1[1] == e2[0]:
            return 0.0
        return self._dijkstra(e1[1], cutoff).get(e2[0], np.inf)

    def _candidates(self, xs, ys):
        out = []
        for x, y in zip(xs, ys):
            p = Point(x, y)
            hits = self.sindex.query(p.buffer(self.radius), predicate="intersects")
            if len(hits) == 0:
                out.append([])
                continue
            cand = [(self.index[i], self.geom[self.index[i]].distance(p))
                    for i in np.atleast_1d(hits)]
            cand.sort(key=lambda t: t[1])
            out.append(cand[: self.k])
        return out

    # ---------------------------------------------------------------- viterbi
    def match(self, lon, lat):
        """Best edge per fix.  Returns (edges, snap_dist, keep_mask).

        A fix with no edge within `radius_m` is dropped from `keep_mask` rather
        than failing the whole trip.
        """
        pts = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326").to_crs(self.crs)
        X, Y = pts.x.values, pts.y.values
        cands = self._candidates(X, Y)

        keep = np.array([len(c) > 0 for c in cands])
        if keep.sum() < 2:
            return None, None, keep
        idx = np.flatnonzero(keep)
        cands = [cands[i] for i in idx]
        X, Y = X[idx], Y[idx]

        score = {e: 0.5 * (d / self.sigma) ** 2 for e, d in cands[0]}
        back = [{}]
        for t in range(1, len(cands)):
            gc = float(np.hypot(X[t] - X[t - 1], Y[t] - Y[t - 1]))
            cutoff = max(gc * self.max_factor, self.slack)
            new, bp = {}, {}
            for e2, d2 in cands[t]:
                best_e, best_s = None, np.inf
                for e1, s1 in score.items():
                    nd = self._net_dist(e1, e2, cutoff)
                    s = s1 + (abs(nd - gc) / self.beta if np.isfinite(nd) else 1e6)
                    if s < best_s:
                        best_s, best_e = s, e1
                new[e2] = best_s + 0.5 * (d2 / self.sigma) ** 2
                bp[e2] = best_e
            score = new
            back.append(bp)

        e = min(score, key=score.get)
        path = [e]
        for t in range(len(cands) - 1, 0, -1):
            e = back[t][e]
            path.append(e)
        path.reverse()
        dist = np.array([dict(c).get(e, np.nan) for c, e in zip(cands, path)])
        return path, dist, keep

    # ------------------------------------------------------------ route fill
    def fill_route(self, path, gc_gaps):
        """Edges strictly between consecutive fixes.  Returns (inserted, ok).

        A detour far longer than the straight-line gap means the two fixes were
        not really matched to the same journey -> the trip is rejected instead
        of being stitched together with an invented loop.
        """
        inserted, ok = [], True
        for (e1, e2), gc in zip(zip(path, path[1:]), gc_gaps):
            if e1 == e2 or e1[1] == e2[0]:
                inserted.append([])
                continue
            try:
                nodes = nx.shortest_path(self.Gp, e1[1], e2[0], weight="length")
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                inserted.append(None)
                ok = False
                continue
            seq = [min(((a, b, k) for k in self.Gp[a][b]), key=lambda e: self.length[e])
                   for a, b in zip(nodes, nodes[1:])]
            if sum(self.length[e] for e in seq) > self.max_factor * gc + self.slack:
                inserted.append(None)
                ok = False
                continue
            inserted.append(seq)
        return inserted, ok

    def route_points(self, lon, lat, times, path, inserted):
        """Weave observed fixes and inferred intermediate edges into one track.

        Observed fixes keep their own coordinates and timestamps; each inserted
        edge contributes one point at its start node, timed by how far along the
        inter-fix route it sits.  `observed` flags which is which.
        """
        out_lon, out_lat = [float(lon[0])], [float(lat[0])]
        out_t, out_e, obs = [float(times[0])], [path[0]], [True]

        for i, seq in enumerate(inserted):
            if seq:
                lens = np.array([self.length[e] for e in seq], float)
                cum = np.cumsum(np.r_[0.0, lens])
                total = cum[-1] if cum[-1] > 0 else 1.0
                t0, t1 = float(times[i]), float(times[i + 1])
                for j, e in enumerate(seq):
                    x, y = self.geom[e].coords[0][:2]
                    a, b = self._to_lonlat(x, y)
                    out_lon.append(float(a)); out_lat.append(float(b))
                    out_t.append(t0 + (cum[j] / total) * (t1 - t0))
                    out_e.append(e); obs.append(False)
            out_lon.append(float(lon[i + 1])); out_lat.append(float(lat[i + 1]))
            out_t.append(float(times[i + 1])); out_e.append(path[i + 1]); obs.append(True)

        return (np.asarray(out_lon), np.asarray(out_lat),
                np.maximum.accumulate(np.asarray(out_t, float)),
                out_e, np.asarray(obs))


def connectivity(seq):
    """Share of consecutive segment changes where the two segments share a node.

    After `fill_route` this should be 1.0 — it is a guard against a broken
    route, no longer a proxy for the sampling rate.
    """
    pairs = [(a, b) for a, b in zip(seq, seq[1:]) if a != b]
    if not pairs:
        return 1.0
    return sum(1 for (_, v1, _), (u2, _, _) in pairs if v1 == u2) / len(pairs)


# --- geometry -----------------------------------------------------------------
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def to_epoch(series):
    """Resolution-safe datetime -> unix seconds (pandas 2 and 3)."""
    tz = getattr(series.dtype, "tz", None)
    zero = pd.Timestamp("1970-01-01", tz=tz) if tz is not None else pd.Timestamp("1970-01-01")
    return ((series - zero) // pd.Timedelta("1s")).astype("int64")

In [ ]:
def segment_trips(df, max_gap_s=300, max_speed_ms=45.0, still_m=25.0, stop_s=180,
                  id_col="vehicle_id", break_col=None):
    """Cut a per-vehicle stream of GPS fixes into trips.

    A new trip starts on a time gap, on a physically impossible jump, after the
    vehicle has stood still (< `still_m` between fixes) for `stop_s`, or wherever
    the boolean column `break_col` is True.  Fixes inside a standstill are dropped.
    """
    df = df.sort_values([id_col, "epoch"], kind="mergesort").reset_index(drop=True)
    dt = df["epoch"].diff()
    dist = haversine_m(df["lon"].shift(), df["lat"].shift(), df["lon"], df["lat"])
    new_vehicle = df[id_col] != df[id_col].shift()
    dt = dt.where(~new_vehicle)
    dist = dist.where(~new_vehicle)
    speed = dist / dt.replace(0, np.nan)

    df["_dt"], df["_dist"], df["_speed"] = dt, dist, speed

    # standstill runs -----------------------------------------------------------
    still = (dist < still_m).fillna(False) & ~new_vehicle
    run = (~still).cumsum()
    run_span = (df.groupby(run)["epoch"].transform("max")
                - df.groupby(run)["epoch"].transform("min"))
    idle = still & (run_span >= stop_s)

    brk = new_vehicle | (dt > max_gap_s) | (dt <= 0) | (speed > max_speed_ms)
    brk = brk | idle.shift(fill_value=False) & ~idle          # first fix after a standstill
    if break_col is not None:                                  # e.g. a new fare
        brk = brk | df[break_col].astype(bool)

    df["trip_key"] = brk.cumsum()
    df = df[~idle].copy()
    return df

## 1. Read the .plt files

In [ ]:
def read_plt(path):
    d = pd.read_csv(path, skiprows=6, header=None,
                    names=["lat", "lon", "zero", "alt", "days", "date", "time"])
    d["ts"] = pd.to_datetime(d["date"] + " " + d["time"], errors="coerce")
    return d[["lat", "lon", "alt", "ts"]]


users = sorted(d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d)))
labelled = [u for u in users if os.path.exists(os.path.join(RAW_DIR, u, "labels.txt"))]
print(f"{len(users)} users, {len(labelled)} of them labelled")

wanted = labelled if not USE_UNLABELLED else users
parts = []
for u in tqdm(wanted, desc="users"):
    for path in glob.glob(os.path.join(RAW_DIR, u, "Trajectory", "*.plt")):
        d = read_plt(path)
        d["user"] = u
        parts.append(d)
raw = pd.concat(parts, ignore_index=True)
del parts
print(raw.shape)
raw.head(3)

In [ ]:
df = raw.dropna(subset=["lat", "lon", "ts"])
df["epoch"] = to_epoch(df["ts"].dt.tz_localize(SOURCE_TZ))
df = df[(df.lon.between(BBOX[0], BBOX[2])) & (df.lat.between(BBOX[1], BBOX[3]))]
df = df.rename(columns={"user": "vehicle_id"}).drop_duplicates(["vehicle_id", "epoch"])
print(f"{len(df)} fixes in the bbox, {df.vehicle_id.nunique()} users, "
      f"{pd.to_datetime(df.epoch.min(), unit='s')} -> {pd.to_datetime(df.epoch.max(), unit='s')}")

## 2. Attach the mode labels

Each row of `labels.txt` is an interval. A fix belongs to the interval that
contains it; fixes outside every interval are unlabelled.

In [ ]:
lab = []
for u in labelled:
    t = pd.read_csv(os.path.join(RAW_DIR, u, "labels.txt"), sep="\t")
    t.columns = [c.strip().lower().replace(" ", "_") for c in t.columns]
    t["vehicle_id"] = u
    lab.append(t)
lab = pd.concat(lab, ignore_index=True)
lab["t0"] = to_epoch(pd.to_datetime(lab["start_time"]).dt.tz_localize(SOURCE_TZ))
lab["t1"] = to_epoch(pd.to_datetime(lab["end_time"]).dt.tz_localize(SOURCE_TZ))
lab["label_id"] = np.arange(len(lab))
print(len(lab), "label intervals")
print(lab.transportation_mode.value_counts().to_string())

In [ ]:
# interval join, per user
pieces = []
for u, g in tqdm(df.groupby("vehicle_id", sort=False), desc="labelling"):
    l = lab[lab.vehicle_id == u].sort_values("t0")
    if l.empty:
        g = g.assign(transportation_mode=None, label_id=np.nan)
    else:
        idx = np.searchsorted(l["t0"].values, g["epoch"].values, side="right") - 1
        ok = (idx >= 0) & (g["epoch"].values <= l["t1"].values[np.clip(idx, 0, None)])
        g = g.assign(
            transportation_mode=np.where(ok, l["transportation_mode"].values[np.clip(idx, 0, None)], None),
            label_id=np.where(ok, l["label_id"].values[np.clip(idx, 0, None)], np.nan))
    pieces.append(g)
df = pd.concat(pieces, ignore_index=True)
print("labelled fixes:", int(df.label_id.notna().sum()), "/", len(df))
print(df.transportation_mode.value_counts().to_string())

## 3. Trips

For a labelled user a trip is one label interval — real boundaries, and a mode
tag with it. Long intervals are still cut on gaps and standstills.

In [ ]:
sel = df[df.transportation_mode.isin(MODES)].copy() if MODES else df.copy()
if USE_UNLABELLED:
    sel = pd.concat([sel, df[df.label_id.isna()]], ignore_index=True)

sel["_label_change"] = (sel["label_id"] != sel.groupby("vehicle_id")["label_id"].shift()).fillna(True)
sel = sel.sort_values(["vehicle_id", "epoch"])

trips = segment_trips(sel, max_gap_s=MAX_GAP_S, max_speed_ms=MAX_SPEED_MS,
                      still_m=STILL_M, stop_s=STOP_S, id_col="vehicle_id",
                      break_col="_label_change")
trips["trip_id"] = trips["vehicle_id"].astype(str) + "_" + trips["trip_key"].astype(str)
print(len(trips), "fixes in", trips.trip_key.nunique(), "candidate trips")

In [ ]:
agg = trips.groupby("trip_key").agg(n=("epoch", "size"), t0=("epoch", "first"),
                                    t1=("epoch", "last"), meters=("_dist", "sum"))
agg["duration"] = agg.t1 - agg.t0
agg["mean_speed"] = agg.meters / agg.duration.replace(0, np.nan)

good = agg[(agg.n >= MIN_POINTS)
           & agg.duration.between(MIN_SECONDS, MAX_SECONDS)
           & (agg.meters >= MIN_METERS)
           & agg.mean_speed.between(*MEAN_SPEED_RANGE)].index
print(f"{len(good)} / {len(agg)} trips pass the filters")
if MAX_TRIPS and len(good) > MAX_TRIPS:
    good = pd.Series(good).sample(MAX_TRIPS, random_state=RANDOM_SEED).values
trips = trips[trips.trip_key.isin(set(good))]
agg.loc[good, ["n", "duration", "meters", "mean_speed"]].describe()

## 4. Road network

In [ ]:
# The Overpass download takes a few minutes and is cached as GraphML afterwards.
if os.path.exists(GRAPHML):
    G = ox.io.load_graphml(GRAPHML)
else:
    G = ox.graph.graph_from_bbox(BBOX, network_type="drive", simplify=True,
                                 truncate_by_edge=True)
    ox.io.save_graphml(G, GRAPHML)

G = ox.truncate.largest_component(G, strongly=True)
print(G)

## 5. Map matching

In [ ]:
uid = assign_unique_osmids(G)
matcher = Matcher(G, radius_m=SNAP_RADIUS_M, sigma_m=SNAP_SIGMA_M,
                  beta_m=BETA_M, max_route_factor=MAX_ROUTE_FACTOR)

rows, observed = [], []
rejected = {"sparse": 0, "no_candidate": 0, "snap": 0,
            "unroutable": 0, "short": 0, "connectivity": 0}

for trip_key, g in tqdm(list(trips.groupby("trip_key", sort=False)), desc="map matching"):
    lon, lat, ts = g["lon"].values, g["lat"].values, g["epoch"].values

    # A route cannot be reconstructed honestly from fixes this far apart:
    # the shortest path between them is a guess, not an observation.
    if len(ts) > 1 and np.median(np.diff(ts)) > MAX_MEDIAN_GAP_S:
        rejected["sparse"] += 1
        continue

    path, snap, keep = matcher.match(lon, lat)
    if path is None:
        rejected["no_candidate"] += 1
        continue
    lon, lat, ts = lon[keep], lat[keep], ts[keep]

    ok = snap <= MAX_SNAP_M
    if ok.mean() < MIN_MATCHED_FRAC:
        rejected["snap"] += 1
        continue
    lon, lat, ts = lon[ok], lat[ok], ts[ok]
    path = [e for e, m in zip(path, ok) if m]
    if len(path) < 2:
        rejected["short"] += 1
        continue

    p = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326").to_crs(matcher.crs)
    gc = np.hypot(np.diff(p.x.values), np.diff(p.y.values))
    inserted, routable = matcher.fill_route(path, gc)
    if not routable:
        rejected["unroutable"] += 1
        continue

    LON, LAT, T, E, obs = matcher.route_points(lon, lat, ts, path, inserted)
    if len(E) < MIN_POINTS or T[-1] - T[0] < MIN_SECONDS:
        rejected["short"] += 1
        continue
    if connectivity(E) < MIN_CONNECTIVITY:
        rejected["connectivity"] += 1
        continue

    stamps = np.maximum.accumulate(np.round(T).astype("int64")).tolist()
    observed.append(float(obs.mean()))
    rows.append({
        "Id": str(g["trip_id"].iloc[0]),
        "Coordinates": str([(round(float(a), 6), round(float(b), 6))
                            for a, b in zip(LON, LAT)]),
        "OSMids": str([uid[e]["unique_osmid"] for e in E]),
        "Timestamps": str(stamps),
        "Total_time": stamps[-1] - stamps[0],
    })

print(f"kept {len(rows)} trips; rejected {rejected}")
if observed:
    obs_mean = float(np.mean(observed))
    print(f"observed share of emitted points: {obs_mean:.1%} "
          f"(the rest are inferred route between fixes)")
    if obs_mean < 0.5:
        print("WARNING: most points are inferred, not observed. The sampling is too\n"
              "         coarse for a trustworthy route — treat this as OD data.")


## 6. Write the gold files

In [ ]:
matched = pd.DataFrame(rows, columns=["Id", "Coordinates", "OSMids", "Timestamps", "Total_time"])
matched.to_csv(f"{OUT}/matched_trips_{CITY}.csv", index=True)

edge_list = build_edge_list_directed(G, uid)
edge_list.to_csv(f"{OUT}/edge_list_directed_{CITY}.csv", index=False)

n_feat = build_road_network_geojson(G, uid, f"{OUT}/road_network_unique_osmids_{CITY}.geojson")
print(len(matched), "trips |", len(edge_list), "transitions |", n_feat, "segments")
matched.head(2)

In [ ]:
# the mode label is the whole point of GeoLife — keep it next to the trips
mode = (trips.groupby("trip_id")["transportation_mode"]
              .agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else None))
mode = mode[mode.index.isin(matched["Id"])]
mode.rename("transportation_mode").to_csv(f"{OUT}/trip_mode_{CITY}.csv")
print(mode.value_counts().to_string())

In [ ]:
def validate_gold(trips_path, edge_list_path=None, geojson_path=None, require_coords=True):
    """Check the produced files against the Harbin/Omsk contract."""
    df = pd.read_csv(trips_path)
    problems = []

    expected = ["Unnamed: 0", "Id", "Coordinates", "OSMids", "Timestamps", "Total_time"]
    if list(df.columns) != expected:
        problems.append(f"columns are {list(df.columns)}, expected {expected}")

    bad_len = bad_eval = bad_total = bad_coord = 0
    osmids_seen = set()
    for _, r in df.iterrows():
        try:
            c = ast.literal_eval(r["Coordinates"])
            o = ast.literal_eval(r["OSMids"])
            t = ast.literal_eval(r["Timestamps"])
        except Exception:
            bad_eval += 1
            continue
        if not (len(c) == len(o) == len(t)):
            bad_len += 1
        if t[-1] - t[0] != r["Total_time"]:
            bad_total += 1
        if require_coords and not all(isinstance(p, tuple) and len(p) == 2 for p in c):
            bad_coord += 1
        osmids_seen.update(map(str, o))

    for label, n in [("rows that do not literal_eval", bad_eval),
                     ("rows with unequal list lengths", bad_len),
                     ("rows where Total_time != Timestamps[-1] - Timestamps[0]", bad_total),
                     ("rows with malformed coordinates", bad_coord)]:
        if n:
            problems.append(f"{n} {label}")

    print(f"{trips_path}: {len(df)} trips, {len(osmids_seen)} distinct segments, "
          f"Total_time median {df['Total_time'].median():.0f} s")

    if edge_list_path:
        el = pd.read_csv(edge_list_path, dtype=str)
        if list(el.columns) != ["osmid_u", "osmid_v"]:
            problems.append(f"edge list columns are {list(el.columns)}")
        known = set(el["osmid_u"]) | set(el["osmid_v"])
        missing = osmids_seen - known
        print(f"{edge_list_path}: {len(el)} transitions, "
              f"{len(osmids_seen & known)}/{len(osmids_seen)} trip segments present")
        if missing and len(missing) > 0.05 * max(len(osmids_seen), 1):
            problems.append(f"{len(missing)} trip segments missing from the edge list")

    if geojson_path:
        with open(geojson_path) as f:
            gj = json.load(f)
        keys = {f["properties"].get("unique_osmid", f["properties"]["osmid"])
                for f in gj["features"]}
        print(f"{geojson_path}: {len(gj['features'])} features, {len(keys)} unique ids")
        if not osmids_seen <= keys:
            problems.append(f"{len(osmids_seen - keys)} trip segments missing from the geojson")

    print("\nOK — matches the gold contract" if not problems
          else "\nPROBLEMS:\n  " + "\n  ".join(problems))
    return df

In [ ]:
_ = validate_gold(f"{OUT}/matched_trips_{CITY}.csv",
                  f"{OUT}/edge_list_directed_{CITY}.csv",
                  f"{OUT}/road_network_unique_osmids_{CITY}.geojson")

## Caveats

* **Mode labels cover 69 of 182 users**, and the mode mix is nothing like a taxi
  fleet: walk and bus dominate, `car`/`taxi` are a minority. Check
  `trip_mode_beijing_geolife.csv` before claiming a car dataset.
* **Beijing 2007–2012 on a 2026 OSM network** — five years of trajectories against
  a network that has changed enormously since. The 5th-ring bbox keeps the worst
  of it out; widen it and matching quality drops.
* **Not vehicle-only.** A `bus` trip includes stops at every stop; a `subway` trip
  is not on the road network at all (excluded by `MODES` here). Do not pool modes
  into one training set without a mode feature.
* **Mixed devices and sampling.** 91 % dense, the rest is whatever the user's
  logger did. The filters drop the worst but the sampling distribution stays wide.
* If our table already has a Beijing column and it is T-Drive, GeoLife is a
  *different* Beijing — say which is which, do not merge them.